In [ ]:
import math
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Any

from statsmodels.distributions.empirical_distribution import ECDF

import matplotlib.pyplot as plt
from matplotlib import rcParams


In [ ]:
male_data = pd.read_csv('male_data.csv')

In [ ]:
male_data = male_data[male_data.columns[3:]]

In [ ]:
#找出所有达标记录
filtered_male = male_data[(male_data['Y染色体浓度'] > 0.04)].copy()
#到每个孕妇最早达标的记录
dabiao_male = filtered_male.groupby('孕妇代码').first().reset_index()

In [ ]:
filtered_male

In [ ]:
dabiao_male

In [ ]:
dabiao_male.to_csv('dabiao_male.csv')

In [ ]:
plt.scatter(dabiao_male['检测孕周'],dabiao_male['孕妇BMI'])

In [ ]:
data1=dabiao_male[["孕妇BMI",'检测孕周']]

In [ ]:
data1.columns=['BMI', 'earliest_time'] 

In [ ]:
data1.to_csv('BMI-earliest.csv')

In [ ]:
df = pd.read_csv("BMI-earliest.csv")

In [ ]:
## 模拟退火

In [ ]:
def sa_bmi_bins(
    df: pd.DataFrame,
    x_col: str = "BMI",
    y_col: str = "earliest_time",
    K: int = 5,                    # 5 组 => 4 个切点
    min_group_frac: float = 0.05,  # 每组最小占比
    T0: float = 1.0,               # 初始温度
    Tend: float = 1e-3,            # 结束温度
    iters: int = 5000,             # 迭代步数（越大越稳）
    neighbor_radius: int = 5,      # 邻域半径（切点索引的最大步长）
    random_seed: int = 42          # 随机种子
) -> Dict[str, Any]:
    """
    Simulated Annealing（模拟退火）在一维（BMI）上寻找 K=5 分组的 4 个切点，最小化 y 的组内 SSE。
    约束：每组样本数 >= ceil(min_group_frac * n)。

    返回：
      {
        "threshold_values": [b1..b4],      # BMI 上的阈值（相邻中点）
        "threshold_indices": [i1..i4],     # 排序后 1-based 右端点索引
        "bin_edges": [minBMI, b1, ..., maxBMI],
        "labels": [(L1,R1), ...],          # 各组 BMI 区间
        "group_stats": pd.DataFrame,       # 每组 n、区间、y_mean、y_var
        "sse": float,                      # 最优 SSE
        "sorted_data": pd.DataFrame        # 按 BMI 排序后的数据
      }
    """
    # 1) 预处理与排序
    dat = df[[x_col, y_col]].dropna().copy()
    if dat.empty:
        raise ValueError("输入数据为空或全为 NaN。")
    dat.sort_values(by=x_col, inplace=True, kind="mergesort")
    x = dat[x_col].to_numpy(dtype=float)
    y = dat[y_col].to_numpy(dtype=float)
    n = len(y)
    if n < K:
        raise ValueError(f"样本量 n={n} < K={K}，无法切分。")

    # 2) 最小组容量（样本数）
    mmin = max(1, math.ceil(min_group_frac * n))
    if mmin * K > n:
        mmin = max(1, n // K)

    # 3) 前缀和（O(1) 计算任意区间 SSE）
    S1 = np.zeros(n + 1)  # sum y
    S2 = np.zeros(n + 1)  # sum y^2
    S1[1:] = np.cumsum(y)
    S2[1:] = np.cumsum(y * y)

    def interval_sse(u: int, v: int) -> float:
        """区间 [u, v]（1-based, 含端点）的 SSE。"""
        m = v - u + 1
        sy = S1[v] - S1[u - 1]
        sy2 = S2[v] - S2[u - 1]
        return float(sy2 - (sy * sy) / m)

    def total_sse(cuts: List[int]) -> float:
        """给定 1-based 右端点切点，计算总 SSE。"""
        boundaries = [0] + cuts + [n]
        tot = 0.0
        for k in range(len(boundaries) - 1):
            u = boundaries[k] + 1
            v = boundaries[k + 1]
            tot += interval_sse(u, v)
        return tot

    def idx_to_thresholds(cuts: List[int]) -> List[float]:
        """1-based 右端点 -> BMI 阈值（相邻中点）。"""
        ths = []
        for i in cuts:
            if 1 <= i < n:
                ths.append(0.5 * (x[i - 1] + x[i]))
            else:
                ths.append(float(x[min(max(i - 1, 0), n - 1)]))
        return ths

    # 4) 可行初解（等人数切分）
    def equal_count_init() -> List[int]:
        cuts = []
        target = n / K
        for g in range(1, K):
            cut = int(round(target * g))
            left_bound  = (cuts[-1] if cuts else 0) + mmin
            right_bound = n - (K - g) * mmin
            cut = int(np.clip(cut, left_bound, right_bound))
            cuts.append(cut)
        return cuts

    # 5) 邻域：随机挑一个切点，移动 [-neighbor_radius, +neighbor_radius] 的整数步
    rng = np.random.default_rng(random_seed)

    def random_neighbor(cuts: List[int]) -> List[int]:
        if not cuts:
            return cuts
        cand = cuts[:]
        k = rng.integers(0, len(cand))          # 随机选一个切点
        i0 = cand[k]
        # 可行边界：保证相邻组 >= mmin
        left_bound  = (cand[k - 1] if k - 1 >= 0 else 0) + mmin
        right_bound = (cand[k + 1] if k + 1 < len(cand) else n) - mmin
        if left_bound > right_bound:
            return cand
        step = int(rng.integers(-neighbor_radius, neighbor_radius + 1))
        if step == 0:
            step = 1 if rng.random() < 0.5 else -1
        new_i = int(np.clip(i0 + step, left_bound, right_bound))
        cand[k] = new_i
        cand.sort()
        return cand

    # 6) 温度调度（指数冷却）
    def temperature(t: int) -> float:
        if iters <= 1: 
            return Tend
        return T0 * (Tend / T0) ** (t / (iters - 1))

    # 7) 主循环：标准 SA（Metropolis）
    cur = equal_count_init()
    cur_val = total_sse(cur)
    best = cur[:]
    best_val = cur_val

    for t in range(iters):
        T = temperature(t)
        cand = random_neighbor(cur)
        cand_val = total_sse(cand)
        delta = cand_val - cur_val
        if delta < 0 or rng.random() < math.exp(-delta / max(T, 1e-12)):
            cur, cur_val = cand, cand_val
            if cur_val < best_val:
                best, best_val = cur[:], cur_val

    # 8) 生成输出
    thr_vals = idx_to_thresholds(best)
    bin_edges = [float(x[0])] + thr_vals + [float(x[-1])]
    labels = [(bin_edges[i], bin_edges[i + 1]) for i in range(len(bin_edges) - 1)]

    # 每组统计
    boundaries = [0] + best + [n]
    rows = []
    for g in range(len(boundaries) - 1):
        u0 = boundaries[g]     # 0-based 左开
        v0 = boundaries[g + 1] # 0-based 右闭
        sl = slice(u0, v0)
        xg, yg = x[sl], y[sl]
        mean = float(yg.mean())
        var  = float(((yg - mean) ** 2).mean()) if len(yg) > 1 else 0.0
        rows.append({
            "group": g + 1,
            "n": int(len(yg)),
            "bmi_left": float(xg.min()),
            "bmi_right": float(xg.max()),
            "y_mean": mean,
            "y_var": var
        })
    stats = pd.DataFrame(rows)

    return {
        "threshold_values": thr_vals,
        "threshold_indices": best,
        "bin_edges": bin_edges,
        "labels": labels,
        "group_stats": stats,
        "sse": float(best_val),
        "sorted_data": dat.reset_index(drop=True)
    }



In [ ]:
df=pd.read_csv('Q3.csv')
df.rename(columns={'孕妇BMI':'BMI','t_theory':'earliest_time'},inplace=True)

In [ ]:
res = sa_bmi_bins(df, x_col="BMI", y_col="earliest_time",
                  K=5, min_group_frac=0.05,
                  T0=1.0, Tend=1e-3, iters=10000, neighbor_radius=5, random_seed=12)
print("BMI阈值（b1..b4）：", res["threshold_values"])
print("各组统计：\n", res["group_stats"])
print(res['sse'])

In [ ]:
k = 0
t = 1
res['sorted_data']['group'] = ''*len(res['sorted_data'])
for i in res['group_stats']['n'].values:
    res['sorted_data']['group'][k:k+i] = [t]*i
    t+=1
    k+=i

In [ ]:
group_male = res['sorted_data']

In [ ]:
group_male.to_csv('group_male.csv')

In [ ]:
# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建函数
def logistic_function(x):
    return 1 / (1 + np.exp(-0.45* (x - 15)))

# 生成数据点
x = np.linspace(0, 40, 500)
y = logistic_function(x)

# 创建图表和轴
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

# 绘制主曲线
ax.plot(x, y, linewidth=3, color='#E63946', label=r'$y=\frac{1}{1+e^{-0.45(x-15)}}$')

# 添加特殊点
point_x = 20
point_y = logistic_function(point_x)
#ax.plot(point_x, point_y, 'o', markersize=8, color='#1D3557', label=f'拐点 ({point_x}, {point_y:.2f})')

# 添加渐近线
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
ax.axhline(y=1, color='gray', linestyle='--', alpha=0.7)

# 设置标题和标签
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)

# 设置坐标轴范围
ax.set_xlim(0, 40)
ax.set_ylim(-0.1, 1.1)

# 添加网格
ax.grid(True, alpha=0.3, linestyle='-')

# 添加图例
ax.legend(loc='lower right', fontsize=12)

# 美化布局
plt.tight_layout()

# 显示图表
plt.show()

In [ ]:
#最小风险点计算
plt.figure(figsize=(12,5))
min_list=[]
for i in range(len(res['group_stats'])):
    group_i = group_male[group_male['group']==i+1].sort_values('earliest_time')
    ecdf = ECDF(group_i.earliest_time)
    def risk(t):
        alpha = 0.6
        beta = 0.4
        risk1 = alpha * (1-ecdf(t))
        risk2 = beta * 1 / (1 + np.exp(-0.37 * (t - 20)))
        return risk1 + risk2
    list1 = np.arange(group_i['earliest_time'].values[0],group_i['earliest_time'].values[-1],0.01)
    list2 = [risk(float(x)) for x in list1]
    min_index = np.argmin(list2)
    min_x = list1[min_index]
    min_y = list2[min_index]
    min_list.append((float(min_x),float(min_y)))
    plt.scatter(min_x, min_y, color='red', zorder=1)  # 红色点标记最低点
    '''
    plt.annotate(f'({min_x:.2f}, {min_y:.2f})',  # 标注坐标
                 xy=(min_x, min_y),  # 标注点位置
                 xytext=(min_x+1, min_y+0.1),  # 文本位置偏移
                 arrowprops=dict(arrowstyle='->', color='red'),  # 箭头属性
                 fontsize=8)
    '''
    plt.plot(list1,list2,label=f"Group{i+1}")
    #plt.text()
    plt.legend()
print(min_list)

In [ ]:
#找出重复检测的,计算Y染色体浓度的测量误差
grouped_male = male_data.groupby(['孕妇代码','检测抽血次数'])

In [ ]:
#重复上述步骤1000次，给每项Y染色体浓度加上误差项，来计算置信区间
min_list_all=[]
for j in range(1000):
    #plt.figure(figsize=(12,5))
    min_list_j=[]
    for i in range(len(res['group_stats'])):
        group_i = group_male[group_male['group']==i+1].sort_values('earliest_time')
        group_i['earliest_time'] = group_i['earliest_time'] + np.random.normal(loc=0,scale=1,size=len(group_i))
        ecdf = ECDF(group_i.earliest_time)
        def risk(t):
            alpha = 0.6
            beta = 0.4
            risk1 = alpha * (1-ecdf(t))
            risk2 = beta * 1 / (1 + np.exp(-0.45 * (t - 15)))
            return risk1 + risk2
        list1 = np.arange(group_i['earliest_time'].values[0],group_i['earliest_time'].values[-1],0.01)
        list2 = [risk(float(x)) for x in list1]
        min_index = np.argmin(list2)
        min_x = list1[min_index]
        min_y = list2[min_index]
        min_list_j.append(float(min_x))
        #plt.scatter(min_x, min_y, color='red', zorder=1)  # 红色点标记最低点
        '''
        plt.annotate(f'({min_x:.2f}, {min_y:.2f})',  # 标注坐标
                     xy=(min_x, min_y),  # 标注点位置
                     xytext=(min_x+1, min_y+0.1),  # 文本位置偏移
                     arrowprops=dict(arrowstyle='->', color='red'),  # 箭头属性
                     fontsize=8)
        '''
        #plt.plot(list1,list2,label=f"Group{i+1}")
        #plt.text()
        #plt.legend()
    min_list_all.append(min_list_j)

In [ ]:
list1=[]
list2=[]
list3=[]
list4=[]
list5=[]
for i in min_list_all:
    list1.append(i[0])
    list2.append(i[1])
    list3.append(i[2])
    list4.append(i[3])
    list5.append(i[4])

In [ ]:
print(np.mean(list1)-1.96*np.std(list1),np.mean(list1)+1.96*np.std(list1))
print(np.mean(list2)-1.96*np.std(list2),np.mean(list2)+1.96*np.std(list2))
print(np.mean(list3)-1.96*np.std(list3),np.mean(list3)+1.96*np.std(list3))
print(np.mean(list4)-1.96*np.std(list4),np.mean(list4)+1.96*np.std(list4))
print(np.mean(list5)-1.96*np.std(list5),np.mean(list5)+1.96*np.std(list5))